In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as wdg
from IPython.display import display

# Load the downloaded UKHSA data
df = pd.read_csv("ukhsa-chart-download.csv")

# Parse the specimen week dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Keep the RSV positivity data for England
rsv_df = df[
    (df["topic"] == "RSV") &
    (df["geography"] == "England") &
    (df["metric"] == "RSV_testing_positivityByWeek")
].copy()

# Remove rows with missing dates or values
rsv_df = rsv_df.dropna(subset=["date", "metric_value"])

# Rename columns for easier plotting
rsv_df = rsv_df.rename(
    columns={
        "date": "week",
        "metric_value": "positivity"
    }
)

# Sort from earliest to latest
rsv_df = rsv_df.sort_values("week").reset_index(drop=True)

# Use calendar years for the first interactive version
rsv_df["year"] = rsv_df["week"].dt.year

print("Number of RSV observations:", len(rsv_df))
print("Date range:", rsv_df["week"].min().date(),
      "to", rsv_df["week"].max().date())

display(rsv_df[["week", "positivity", "year"]].head())

Number of RSV observations: 483
Date range: 2017-07-03 to 2026-09-28


,week,positivity,year
0,2017-07-03,0.29,2017
1,2017-07-10,0.46,2017
2,2017-07-17,0.40,2017
3,2017-07-24,0.81,2017
4,2017-07-31,1.17,2017


In [3]:
year = wdg.Dropdown(
    options=sorted(rsv_df["year"].unique()),
    value=int(rsv_df["year"].max()),
    description="Calendar year:"
)

def rsv_graph(selected_year):
    data = rsv_df[rsv_df["year"] == selected_year]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(
        data["week"],
        data["positivity"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label="RSV"
    )

    ax.set_title(
        f"Weekly RSV PCR positivity in England: {selected_year}"
    )
    ax.set_xlabel("Week beginning (specimen date)")
    ax.set_ylabel("RSV PCR positivity (%)")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

graph = wdg.interactive_output(
    rsv_graph,
    {"selected_year": year}
)

display(year, graph)

Dropdown(description='Calendar year:', index=9, options=(np.int32(2017), np.int32(2018), np.int32(2019), np.in…

Output()

In [5]:
# Latest and peak RSV positivity statistics
latest = rsv_df.loc[rsv_df["week"].idxmax()]
peak = rsv_df.loc[rsv_df["positivity"].idxmax()]

print("RSV OVERVIEW — ENGLAND")
print("-" * 35)
print(f"Latest reporting week: {latest['week'].date()}")
print(f"Latest RSV positivity: {latest['positivity']:.2f}%")
print(f"Highest recorded positivity: {peak['positivity']:.2f}%")
print(f"Week of highest positivity: {peak['week'].date()}")
print(f"Total weekly observations: {len(rsv_df)}")

RSV OVERVIEW — ENGLAND
-----------------------------------
Latest reporting week: 2026-09-28
Latest RSV positivity: 0.51%
Highest recorded positivity: 25.35%
Week of highest positivity: 2018-11-26
Total weekly observations: 483


In [4]:
# Assign each observation to a season running July to June
rsv_df["season_start"] = rsv_df["week"].dt.year.where(
    rsv_df["week"].dt.month >= 7,
    rsv_df["week"].dt.year - 1
)

rsv_df["season"] = (
    rsv_df["season_start"].astype(str)
    + "/"
    + (rsv_df["season_start"] + 1).astype(str)
)

# Week position within the July-to-June season
rsv_df["season_week"] = (
    ((rsv_df["week"].dt.dayofyear
      - pd.to_datetime(rsv_df["week"].dt.year.astype(str) + "-07-01")
        .dt.dayofyear) % 365)
    // 7
) + 1

print("Available RSV seasons:")
print(sorted(rsv_df["season"].unique()))

Available RSV seasons:
['2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026', '2026/2027']


In [5]:
# Calculate the season week consistently using Monday-based reporting weeks
season_start_date = pd.to_datetime(
    rsv_df["season_start"].astype(str) + "-07-01"
)

# Find the Monday on or before 1 July for each season
season_start_monday = (
    season_start_date
    - pd.to_timedelta(season_start_date.dt.weekday, unit="D")
)

rsv_df["season_week"] = (
    (rsv_df["week"] - season_start_monday).dt.days // 7
) + 1

# Sort seasons chronologically
seasons = sorted(rsv_df["season"].unique())

# Default to the four most recent completed seasons
latest_season_start = rsv_df["season_start"].max()
completed_seasons = [
    s for s in seasons
    if int(s.split("/")[0]) < latest_season_start
]
default_seasons = tuple(completed_seasons[-4:])

# Allow users to select seasons to compare
season_selector = wdg.SelectMultiple(
    options=seasons,
    value=default_seasons,
    description="Seasons:",
    rows=min(8, len(seasons)),
    layout=wdg.Layout(width="350px")
)

def plot_season_comparison(selected_seasons):
    if not selected_seasons:
        print("Select at least one season to display the chart.")
        return

    fig, ax = plt.subplots(figsize=(11, 5))

    for season in selected_seasons:
        season_data = rsv_df[
            rsv_df["season"] == season
        ].sort_values("season_week")

        ax.plot(
            season_data["season_week"],
            season_data["positivity"],
            linewidth=2,
            label=season
        )

    ax.set_title("RSV PCR Positivity by Season in England")
    ax.set_xlabel("Week of season (July to June)")
    ax.set_ylabel("RSV PCR positivity (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(title="RSV season")
    plt.tight_layout()
    plt.show()

season_graph = wdg.interactive_output(
    plot_season_comparison,
    {"selected_seasons": season_selector}
)

display(season_selector, season_graph)

SelectMultiple(description='Seasons:', index=(5, 6, 7, 8), layout=Layout(width='350px'), options=('2017/2018',…

Output()